if 'success_flag' in df_clean.columns:
    df_clean['success_flag'] = df_clean['success_flag'].astype(int)

In [39]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.decomposition import PCA

import joblib

In [40]:
df = pd.read_csv('../data/processed/feature_engineered.csv')

In [41]:
# Drop non-numeric / useless columns
drop_cols = [
    'user_id',
    'role',
    'resource_type',
    'action',
    'timestamp',
]

df_clean = df.drop(columns=drop_cols, errors='ignore')

# Convert boolean to int (if exists)
if 'success_flag' in df_clean.columns:
    df_clean['success_flag'] = df_clean['success_flag'].astype(int)

df_clean.head()

,session_duration,access_volume,success_flag,assigned_resource_count,actively_used_resource_count,hour,day_of_week,week,month,date,...,weekend_activity_ratio_zscore,access_time_variance_zscore,weekly_access_change_zscore,access_spike_score_zscore,privilege_usage_gap_zscore,privilege_usage_ratio_zscore,resource_access_concentration_zscore,raw_risk_score,governance_risk_score,risk_category
0,42.0,17.0,1,9,8,0,0,1,1,2024-01-01,...,0.035673,0.092027,0.687101,0.136695,-0.894655,1.138516,-0.100936,0.342899,64.831138,High
1,57.0,14.0,1,7,5,0,0,1,1,2024-01-01,...,-0.675828,-0.214786,-0.046950,0.059383,-0.391897,0.256122,-0.183580,-0.212430,32.561941,Medium
2,61.0,20.0,1,7,5,1,0,1,1,2024-01-01,...,-1.161129,-0.363862,-0.765391,-1.542238,-0.545113,0.529781,0.542003,0.218229,57.586771,Medium
3,31.0,7.0,1,5,4,1,0,1,1,2024-01-01,...,-1.261399,0.569846,-0.387880,-0.023806,-1.259104,0.956613,-0.570736,-0.307262,27.051393,Low
4,75.0,16.0,1,9,7,2,0,1,1,2024-01-01,...,-0.472956,-0.007141,0.125140,-0.538831,-0.391897,0.576993,0.157172,0.132082,52.580917,Medium


In [42]:
df_clean = df_clean.fillna(df_clean.mean(numeric_only=True))

In [43]:
drop_cols = [
    'user_id',
    'role',
    'resource_type',
    'action',
    'timestamp',
    'risk_category'
]

df_clean = df.drop(columns=drop_cols, errors='ignore')

In [44]:
if 'success_flag' in df_clean.columns:
    df_clean['success_flag'] = df_clean['success_flag'].astype(int)

In [45]:
import numpy as np

df_clean = df_clean.select_dtypes(include=[np.number])

In [46]:
print(df_clean.shape)
print(df_clean.isnull().sum())

(12665, 37)
session_duration                        0
access_volume                           0
success_flag                            0
assigned_resource_count                 0
actively_used_resource_count            0
hour                                    0
day_of_week                             0
week                                    0
month                                   0
avg_daily_access                        0
export_ratio                            0
unique_resources                        0
avg_session_duration                    0
is_night                                0
night_access_pct                        0
is_weekend                              0
weekend_activity_ratio                  0
access_time_variance                    0
weekly_access_change                    0
access_spike_score                      0
privilege_usage_gap                     0
privilege_usage_ratio                   0
resource_access_concentration           0
avg_daily_access_zscor

In [47]:
df_clean = df_clean.fillna(df_clean.mean())

In [48]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_clean)

print(X_scaled[:5])

[[-3.81586190e-01 -1.05907541e-01  2.29558705e-01  2.22400187e-01
   1.14931241e+00 -1.87669723e+00 -1.51694432e+00 -1.66176372e+00
  -1.62949289e+00  2.64336072e-01  2.11972615e-01  1.14931241e+00
   9.60329872e-02  1.53112243e+00  6.66271462e-01 -6.18580570e-01
   1.99608289e-02  5.15245247e-01  3.63507908e-01 -1.74831714e-02
  -1.12425674e+00  1.41149781e+00 -1.97790234e-01  1.44775551e-02
   1.00801315e+00  1.07451960e+00 -3.49172175e-01  1.02121632e+00
   3.56796339e-02  9.20452784e-02  6.87236576e-01  1.36722072e-01
  -8.94831410e-01  1.13874050e+00 -1.00955623e-01  1.09089691e+00
   1.09089691e+00]
 [ 4.67502974e-01 -4.43204860e-01  2.29558705e-01 -6.34520521e-01
  -2.07135421e-01 -1.87669723e+00 -1.51694432e+00 -1.66176372e+00
  -1.62949289e+00  1.08993402e-01 -1.88095746e-01 -2.07135421e-01
   2.82202871e-02  1.53112243e+00  5.13773933e-01 -6.18580570e-01
  -2.24978122e-01  4.44822203e-01  6.13271034e-02 -9.35478795e-02
  -5.68469568e-01  4.05018473e-01 -2.88490073e-01 -4.6958

In [49]:
from sklearn.ensemble import IsolationForest

model = IsolationForest(
    n_estimators=100,
    contamination=0.05,
    random_state=42
)

model.fit(X_scaled)

,n_estimators,100
,max_samples,'auto'
,contamination,0.05
,max_features,1.0
,bootstrap,False
,n_jobs,None
,random_state,42
,verbose,0
,warm_start,False


In [50]:
df['anomaly_score'] = model.decision_function(X_scaled)
df['anomaly_label'] = model.predict(X_scaled)

df[['anomaly_score', 'anomaly_label']].head()

,anomaly_score,anomaly_label
0,0.071284,1
1,0.112437,1
2,0.035352,1
3,0.046428,1
4,0.113562,1


In [51]:
df['ml_risk_score'] = (
    (df['anomaly_score'] - df['anomaly_score'].min()) /
    (df['anomaly_score'].max() - df['anomaly_score'].min())
) * 100

In [52]:
df[['ml_risk_score']].describe()

,ml_risk_score
count,12665.000000
mean,60.735809
std,17.718196
min,0.000000
25%,47.508295
50%,63.454669
75%,74.472635
max,100.000000


In [57]:
df.to_csv("processed_user_risk_data.csv", index=False)

In [58]:
import joblib

joblib.dump(model, "isolation_forest_model.pkl")

['isolation_forest_model.pkl']

In [ ]:
joblib.dump(scaler, "scaler.pkl")

['scaler.pkl']

: 